# Clustering SLiMFinder SLiMs according to NormIC

NormIC > 0.4 --> real connection 

## command line code:

python ~/SLiMSuite/tools/comparimotif_V3.py motifs=~/all_idrs.txt searchdb=~/elm_class.txt out=clustering_results_trial.csv


The txt files I made myself by copying the Pattern column from the slimfinder.csv and the Regex column from elm_classes

In [ ]:
!pip install xlsxwriter
import pandas as pd
import io
import re
import requests
import time

In [ ]:
def get_uniprot_data(file_path, column_name='gene name'):
    """Getting the amino acid sequence of relevant proteins"""
    df_input = pd.read_excel(file_path) 
    # Clean the gene names (remove whitespace)
    gene_list = df_input[column_name].dropna().unique().tolist()

    results = []
    print(f"Starting retrieval for {len(gene_list)} unique genes...")

    for gene in gene_list:
        gene = str(gene).strip()
        print(f"Processing: {gene}")
        
        # Querying UniProt specifically for M. tuberculosis H37Rv (TaxID: 83332)
        url = f"https://rest.uniprot.org/uniprotkb/search?query={gene}+AND+taxonomy_id:83332&fields=accession,gene_names,sequence"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                if data['results']:
                    entry = data['results'][0]
                    
                    # Extract Data
                    acc = entry['primaryAccession']
                    # Get all gene names/locus tags associated with this entry
                    gene_data = entry.get('genes', [{}])[0]
                    primary_name = gene_data.get('geneName', {}).get('value', 'N/A')
                    
                    # Get the list of all locus tags (Rv numbers)
                    locus_tags = [lt.get('value') for lt in gene_data.get('orderedLocusNames', [])]
                    locus_str = ", ".join(locus_tags) if locus_tags else "N/A"
                    
                    sequence = entry['sequence']['value']
                    
                    results.append({
                        'Input_Gene': gene,
                        'UniProt_ACC': acc,
                        'Primary_Gene_Name': primary_name,
                        'Locus_Tags': locus_str,
                        'Sequence': sequence
                    })
                else:
                    print(f"  No UniProt match for {gene}")
            else:
                print(f"  API Error {response.status_code} for {gene}")
        except Exception as e:
            print(f"  Failed {gene}: {e}")
        
        # Rate limiting sleep
        time.sleep(0.3)

    df_output = pd.DataFrame(results)
    return df_output


def label_esx1(row):
    "Create column labelling ESX1 proteins"
    # Check both the name UniProt found and the Locus Tags it found
    name = str(row['Gene']).lower()
    if any(p.lower() in name for p in esx1_list):
        return True
    return False


def extract_coords(coord_string):
    "Extract Start and End from 'position_IDR' using Regex"
    match = re.search(r'(\d+)-(\d+)', str(coord_string))
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

In [ ]:
# Load and clean the slimfinder CSV manually to handle the problematic formatting
with open(r"D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/comparimotif/slimfinder.csv", 'r', encoding='utf-8') as f:
    lines = f.readlines()
cleaned_lines = []
for line in lines:
    line = line.strip()
    # Remove outer quotes if the whole line is wrapped in them as formatting of slimfinder csv motif has some lines with quotation
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1].replace('""', '"')
    cleaned_lines.append(line)
master_df = pd.read_csv(io.StringIO("\n".join(cleaned_lines)))
# Load CompariMotif results which compares motifs to ELM motifs for functional identification
compari_df = pd.read_csv('D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/comparimotif/to elm/all_idrs-elm_class.compare.tdt', sep='\t')
elm_function = pd.read_excel('D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/comparimotif/to elm/elm_classes2.xlsx')
# Create the 'Clean_Motif' column to eliminate variants
compari_df['Clean_Motif'] = compari_df['Name2'].str.replace(r'_[a-z]$', '', regex=True, case=False)
# Strip any trailing whitespace just in case
compari_df['Clean_Motif'] = compari_df['Clean_Motif'].str.strip()
elm_function_motifs = elm_function['Regex'].unique().tolist()
elm_motifs_v = compari_df['Clean_Motif'].unique().tolist()
is_contained = set(elm_motifs_v).issubset(set(elm_function_motifs))
print(is_contained) #check all motifs are present in the functional df
all_motifs = master_df['Pattern'].unique().tolist()
elm_motifs = compari_df['Clean_Motif'].unique().tolist() 


True


# Importing relevant protein sequences

In [ ]:
df_output = get_uniprot_data("D:/2nd  year/2nd term/Thesis/files from klaas/unique mtb genes.xlsx", column_name='gene name')

Starting retrieval for 711 unique genes...
Processing: Rv0341
Processing: Rv1975
Processing: Rv2164c
Processing: Rv2131c
Processing: Rv1109c
Processing: Rv2163c
Processing: Rv2383c
Processing: Rv2329c
Processing: Rv0126
Processing: Rv2585c
Processing: Rv3878
Processing: Rv0160c
Processing: Rv0017c
Processing: Rv0280
Processing: Rv3476c
Processing: Rv3882c
Processing: Rv1937
Processing: Rv0449c
Processing: Rv2700
Processing: Rv2770c
Processing: Rv2725c
Processing: Rv3864
Processing: Rv2124c
Processing: Rv3300c
Processing: Rv1039c
Processing: Rv1285
Processing: Rv2777c
Processing: Rv0726c
Processing: Rv3097c
Processing: Rv0556
Processing: Rv1475c
Processing: Rv2841c
Processing: Rv2931
Processing: Rv1664
Processing: Rv1886c
Processing: Rv1427c
Processing: Rv3696c
Processing: Rv2369c
Processing: Rv2125
Processing: Rv2846c
Processing: Rv3630
Processing: Rv1087
Processing: Rv0695
Processing: Rv1351
Processing: Rv2147c
Processing: Rv3479
Processing: Rv3129
Processing: Rv2030c
Processing: Rv34

# Align the motif hubs to the proteins

Taking the motifs that were identified by SLiMFinder and aligning them to the proteins. Also labelling whether the match happens in an IDR (by the known coordinates)

In [ ]:
df_idrs = pd.read_excel('D:/2nd  year/2nd term/Thesis/files from klaas/genetic_variation/analysis_addedmetadata/overview_idrs_with_coords.xlsx') 

final_hits = []

print(f"Starting mapping for {len(df_output)} proteins...")

for _, prot_row in df_output.iterrows():
    gene = prot_row['Input_Gene']
    protein = prot_row['Primary_Gene_Name']
    full_seq = str(prot_row['Sequence'])  
    # Ensure case-insensitivity and remove whitespace
    gene_clean = str(gene).strip().upper()
    df_idrs['gene_names_clean'] = df_idrs['gene_name'].str.strip().str.upper()
    gene_idr_subset = df_idrs[df_idrs['gene_names_clean'] == gene_clean]
    if gene_idr_subset.empty:
        continue
    for pattern in all_motifs: # all unique motif patterns == all_motifs
        try:
            for match in re.finditer(f"(?=({pattern}))", full_seq):
                m_match_seq = match.group(1)
                m_start = match.start() + 1
                m_end = m_start + len(m_match_seq) - 1
                
                match_is_in_idr = False
                idr_range = "N/A"
                
                for _, idr_row in gene_idr_subset.iterrows():
                    if m_start >= idr_row['start'] and m_end <= idr_row['end']:
                        match_is_in_idr = True
                        idr_range = f"{idr_row['start']}-{idr_row['end']}"
                        break 
                
                final_hits.append({
                    'Gene': gene,
                    'Protein': protein, 
                    'Motif_Pattern': pattern,
                    'Sequence_Match': m_match_seq,
                    'Motif_Start': m_start,
                    'Motif_End': m_end,
                    'Is_in_IDR': match_is_in_idr,
                    'IDR_Coordinates': idr_range
                })
        except:
            continue
if not final_hits:
    print("CRITICAL ERROR: No motifs were found in any protein sequences. Check if 'all_motifs' or 'df_output' is empty.")
else:
    df_results = pd.DataFrame(final_hits)
    df_slims_only = df_results[df_results['Is_in_IDR'] == True].copy()
    
    if df_slims_only.empty:
        print("No motifs were found inside the specified IDR coordinates.")

Starting mapping for 711 proteins...


# Check if they're in the ESX-1 proteins

In [ ]:
# Define ESX-1 protein list
esx1_list = [
    "Rv0757", "Rv0758", "Rv0981", "Rv0982", "Rv3597c", "Rv3614c", 
    "Rv3615c", "Rv3616c", "Rv3849", "Rv3862c", "Rv3863", "Rv3864",
    "Rv3865", "Rv3866", "Rv3867", "Rv3868", "Rv3869", "Rv3870", 
    "Rv3871", "Rv3872", "Rv3873", "Rv3874", "Rv3875", "Rv3876",
    "Rv3877", "Rv3878", "Rv3879c", "Rv3880c", "Rv3881c", "Rv3882c", "Rv3883c"
    ]
df_slims_only['Is_ESX1'] = df_slims_only.apply(label_esx1, axis=1)

# Are the motifs in the ESX-1 conserved IDRs from IDRs_ESX.xlsx

In [ ]:
df_conserved = pd.read_excel('D:/2nd  year/2nd term/Thesis/motifs_attempts/all_mtb_genes_from klaas/SLiMFinder/results_allIDRs_oldpython_blast_final_noextras/New folder/ESX-1_GOIs_edited.xlsx')
df_conserved[['cons_start', 'cons_end']] = df_conserved['position_IDR'].apply(lambda x: pd.Series(extract_coords(x)))
df_conserved['gene'] = df_conserved['gene'].astype(str).str.strip().str.upper()
detailed_matches = []
df_slims_only['in_conserved_esx_idr'] = False
for idx, motif_row in df_slims_only.iterrows():
    m_gene = str(motif_row['Gene']).strip().upper()
    m_start = motif_row['Motif_Start']
    m_end = motif_row['Motif_End']
    gene_targets = df_conserved[df_conserved['gene'] == m_gene]
    for _, cons_row in gene_targets.iterrows():
        c_start = cons_row['cons_start']
        c_end = cons_row['cons_end']
        if m_start >= c_start and m_end <= c_end:
            df_slims_only.at[idx, 'in_conserved_esx_idr'] = True
            detailed_matches.append({
                'Gene': m_gene,
                'Motif_Pattern': motif_row['Motif_Pattern'],
                'Motif_Sequence': motif_row['Sequence_Match'],
                'Motif_Coords': f"{m_start}-{m_end}",
                'Conserved_IDR_Coords': cons_row['position_IDR']
            })
df_conserved_mapping_details = pd.DataFrame(detailed_matches)

In [41]:
print(df_conserved_mapping_details)

       Gene   Motif_Pattern Motif_Sequence Motif_Coords Conserved_IDR_Coords
0    RV3878     [HKR]P[AGS]            KPA      196-198              176-280
1    RV3878     [HKR]P[AGS]            RPA      257-259              176-280
2    RV3878             LAG            LAG      102-104               97-113
3    RV3864   [ILV][ST][DE]            VSD      368-370              361-402
4   RV3616C     [HKR]P[AGS]            HPS      346-348              301-383
5   RV3616C             LAG            LAG      279-281              270-294
6   RV3881C  [AGS]..[AGS].$          SKESK      456-460              453-460
7   RV3881C   [ST]..[AGS].$          SKESK      456-460              453-460
8   RV3881C     [AGS][HKR]$             SK      459-460              453-460
9   RV3881C     [HKR]P[AGS]            RPA      384-386              352-433
10  RV3879C     [HKR]P[AGS]            KPG      252-254              175-461
11  RV3879C     [HKR]P[AGS]            KPA      329-331              175-461

In [ ]:
# Show only the 'True' hits
conserved_hits = df_slims_only[df_slims_only['in_conserved_esx_idr']]
print(conserved_hits)
conserved_hits.to_excel('conserved_hits.xlsx')

         Gene Protein   Motif_Pattern Sequence_Match  Motif_Start  Motif_End  \
77     Rv3878    espJ     [HKR]P[AGS]            KPA          196        198   
78     Rv3878    espJ     [HKR]P[AGS]            RPA          257        259   
80     Rv3878    espJ             LAG            LAG          102        104   
132    Rv3864    espE   [ILV][ST][DE]            VSD          368        370   
2543  Rv3616c    espA     [HKR]P[AGS]            HPS          346        348   
2544  Rv3616c    espA             LAG            LAG          279        281   
2950  Rv3881c    espB  [AGS]..[AGS].$          SKESK          456        460   
2951  Rv3881c    espB   [ST]..[AGS].$          SKESK          456        460   
2954  Rv3881c    espB     [AGS][HKR]$             SK          459        460   
2955  Rv3881c    espB     [HKR]P[AGS]            RPA          384        386   
2959  Rv3879c    espK     [HKR]P[AGS]            KPG          252        254   
2960  Rv3879c    espK     [HKR]P[AGS]   

# Add comparimotif data to the conserved MTB motifs

In [ ]:

compari_df['Desc2'] = compari_df['Desc2'].astype(str)
primary_matches = compari_df[~compari_df['Desc2'].str.contains('variant', case=False)]
hits_compari_matches = pd.merge(
    conserved_hits, 
    compari_df[['Name1', 'Clean_Motif', 'NormIC', 'MatchIC']],
    left_on='Motif_Pattern', 
    right_on='Name1', 
    how='left'
)
annotated_comparimotif = pd.merge(
    hits_compari_matches,
    elm_function,
    left_on='Clean_Motif',
    right_on='Regex',
    how='left'
)
print(annotated_comparimotif)
annotated_comparimotif.to_excel("ESX1_Appendix_final.xlsx", index=False)
output_file = "ESX1_Annotated_Motifs_by_Gene.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    unique_genes = annotated_comparimotif['Gene'].unique()
    for gene in unique_genes:
        gene_df = annotated_comparimotif[annotated_comparimotif['Gene'] == gene]
        gene_df.to_excel(
            writer, 
            sheet_name=str(gene)[:31], 
            index=False
        )
        worksheet = writer.sheets[str(gene)[:31]]
        worksheet.freeze_panes(1, 0)  # Freeze the header row
        worksheet.autofilter(0, 0, len(gene_df), len(gene_df.columns) - 1)
print(f"Successfully created {output_file} with {len(unique_genes)} gene-specific sheets.")

Defaulting to user installation because normal site-packages is not writeable
       Gene Protein Motif_Pattern Sequence_Match  Motif_Start  Motif_End  \
0    Rv3878    espJ   [HKR]P[AGS]            KPA          196        198   
1    Rv3878    espJ   [HKR]P[AGS]            KPA          196        198   
2    Rv3878    espJ   [HKR]P[AGS]            KPA          196        198   
3    Rv3878    espJ   [HKR]P[AGS]            KPA          196        198   
4    Rv3878    espJ   [HKR]P[AGS]            KPA          196        198   
..      ...     ...           ...            ...          ...        ...   
487  Rv3863     N/A   ^..[AG][DE]           MAGE            1          4   
488  Rv3863     N/A   ^..[AG][DE]           MAGE            1          4   
489  Rv3863     N/A   ^..[AG][DE]           MAGE            1          4   
490  Rv3863     N/A   ^..[AG][DE]           MAGE            1          4   
491  Rv3863     N/A   ^..[AG][DE]           MAGE            1          4   

     Is_i